# DSv4 Pallas-TPU kernel bench on Colab (v5e-1)

Runs the DSv4 attention kernels from [`HyperBlaze456/auto-jax-kernel`](https://github.com/HyperBlaze456/auto-jax-kernel) on a single TPU v5e chip.

**Before running:** `Runtime > Change runtime type > TPU` (Colab's TPU runtime is a single **v5e-1** chip, 16 GB HBM). Then `Runtime > Run all`.

The kernels target **jax 0.10.1 + Pallas**, so we pin that exact version — Pallas API / TPU lowering can drift across JAX releases, so a known-good pin keeps the bench reproducible.

> **Why we don't `!python bench.py`.** A TPU chip can be owned by **exactly one process** at a time. This Colab kernel claims the chip the moment it calls `jax.devices()` (cell 2). Shelling out with `!python bench.py …` would launch a *second* process that finds the TPU busy and dies with `Unable to initialize backend 'tpu'`. So instead we **import `bench.py` and call it in-process** — one kernel, one TPU owner. Cells below use a thin `run(...)` / `sweep(...)` driver that is a 1:1 stand-in for the `bench.py` CLI.
>
> **Iterating on the kernel:** because the code is imported once into this kernel, after you `git pull` new code (cell 3) you must `Runtime > Restart session` and `Run all` to pick it up.

In [ ]:
# 1. Install the JAX the kernels were written against, with TPU support.
#    Run this FIRST, before importing jax. If a later cell errors with a
#    libtpu / 'Unable to initialize backend tpu' message, do
#    Runtime > Restart session, then Run all again (skip re-installing).
#    libtpu (0.0.41) ships as a normal PyPI dep of jax[tpu] now, so no find-links needed.
!pip install -q "jax[tpu]==0.10.1"

In [ ]:
# 2. Confirm we're on a TPU v5e and JAX sees exactly one chip.
import jax
print("jax", jax.__version__, "| jaxlib", jax.lib.__version__)
devs = jax.devices()
print("devices:", devs)
assert devs, "No JAX devices — did you pick the TPU runtime? Runtime > Change runtime type > TPU"
kind = getattr(devs[0], "device_kind", "")
print("device_kind:", kind)
assert "v5e" in kind.lower() or "tpu" in kind.lower(), f"Expected a TPU, got {kind!r}"

In [ ]:
# 3. Get the code. Clones the latest dsv4-kernel branch (idempotent: re-run pulls).
#    NOTE: this fetches whatever is pushed to origin — push your local commits first.
import os
REPO = "https://github.com/HyperBlaze456/auto-jax-kernel.git"
BRANCH = "dsv4-kernel"
if not os.path.isdir("/content/auto-jax-kernel"):
    !git clone -b {BRANCH} {REPO} /content/auto-jax-kernel
else:
    !git -C /content/auto-jax-kernel fetch --quiet origin && git -C /content/auto-jax-kernel checkout {BRANCH} && git -C /content/auto-jax-kernel pull --ff-only
%cd /content/auto-jax-kernel
!git log --oneline -3

### Alt: upload instead of clone
If you'd rather not push (private WIP), skip cell 3 and instead run `tar czf dsv4.tgz dsv4 bench.py` locally, then in a new cell:
```python
from google.colab import files; files.upload()   # pick dsv4.tgz
!mkdir -p /content/auto-jax-kernel && tar xzf dsv4.tgz -C /content/auto-jax-kernel
%cd /content/auto-jax-kernel
```

In [ ]:
# 4b. In-process bench driver.
#     A TPU chip allows only ONE owning process. This kernel already took the
#     chip in cell 2, so `!python bench.py` (a second process) would fail with
#     "Unable to initialize backend 'tpu'". We import bench.py and call it here
#     instead. `run(...)` / `sweep(...)` are 1:1 stand-ins for the bench.py CLI.
import sys
if "/content/auto-jax-kernel" not in sys.path:
    sys.path.insert(0, "/content/auto-jax-kernel")   # so `import bench` / `dsv4` resolve
import jax.numpy as jnp
import bench

_DTYPES = {"float32": jnp.float32, "bfloat16": jnp.bfloat16, "float16": jnp.float16}
_PEAK, _PEAK_LABEL = bench.detect_peak_tflops(None)   # reuses the chip this kernel holds
print(f"peak_tflops auto-detected: {_PEAK:.1f} ({_PEAK_LABEL})")

def run(preset, seq, *, batch=1, bwd=False, dtype="bfloat16", tol=1e-2, seed=0, peak_tflops=None):
    """In-process `python bench.py --preset <preset> --seq <seq> [--bwd ...]`."""
    peak, label = (peak_tflops, f"override={peak_tflops}") if peak_tflops else (_PEAK, _PEAK_LABEL)
    bench.run_one(preset, batch, seq, bwd, _DTYPES[dtype], peak, label, tol, seed)

def sweep(*, batch=1, bwd=False, dtype="bfloat16", tol=1e-2, seed=0):
    """In-process `python bench.py --sweep` (reuses bench.py's preset x seq ladder)."""
    from types import SimpleNamespace
    args = SimpleNamespace(sweep=True, batch=batch, bwd=bwd, tol=tol, seed=seed)
    bench._do(args, _DTYPES[dtype], _PEAK, _PEAK_LABEL)

In [ ]:
# 5. Correctness sanity (tiny shape). Expect `status: pass`.
run("small_csa", 64)
run("small_hca", 64)

In [ ]:
# 6. Forward benchmarks. peak_tflops auto-detects to 197 (TPU v5e) -> MFU is meaningful.
#    Starting conservative for 16 GB HBM; scale seq up until you hit OOM.
run("dsv4_flash_csa", 4096)
run("dsv4_flash_hca", 2048)
# Scale-up (uncomment as the chip allows; the eager ref materializes full K/logits):
# run("dsv4_flash_csa", 8192)
# run("dsv4_flash_csa", 16384)
# run("dsv4_flash_hca", 16384)

In [ ]:
# 7. (optional) Forward + backward timing.
# run("dsv4_flash_csa", 4096, bwd=True)

In [ ]:
# 8. (optional) Full sweep across presets x a seq ladder.
#    On a single v5e-1 the dsv4_pro_* presets (d=7168, n_h=128) and seq>=65536
#    will likely OOM; the sweep logs those as ERROR and continues.
# sweep()